# Capstone 7: generator bake-off on Kaggle

**Purpose.** Run the same six targets through three generators and save the raw candidates: the re-created Capstone 5 TinyLlama adapter, Gemma 4 E4B (prompted) and Claude Haiku (prompted). **This notebook trains nothing.** It only generates text. Only small files come back to the project: one JSONL file per backend, a summary and an environment report.

**Kaggle settings**
1. **Accelerator:** GPU T4 x2 (Gemma 4 E4B is about 16 GB in fp16 and is split over the two GPUs). **Internet:** on (model downloads and the Haiku call).
2. **Datasets attached:** the data dataset (`final_dataset.csv`, `ict_curriculum.md`, `taxonomy.csv`) and a private dataset for this project, for example `capstone7`, holding only these files, **one copy of each**: `src/data.py`, `src/generator.py`; `config/generators.json`; `scripts/bakeoff_runner.py`; `models/c5_adapter/adapter_config.json` and `models/c5_adapter/adapter_model.safetensors`. (`src/__init__.py` is optional; the adapter's tokenizer files and README are not needed.) If you upload a new version, replace the files instead of adding second copies: the notebook uses the first copy it finds.
3. **Secrets attached:** `ANTHROPIC_API_KEY` (required for Haiku). `HF_TOKEN` is optional and only needed if a model page asks you to accept terms.

**Order:** run the cells top to bottom. The smoke test (one candidate per backend) comes before the full run, so a loading problem costs minutes. Gemma is tested last because it is the riskiest.

## 1. Install

The same line as in Capstone 5: the preinstalled `torchao` is older than what the installed `transformers` and `peft` expect (harmless "Failed to load ... .so" warnings may follow). `anthropic` is the SDK used for Haiku, as in the key test.

In [1]:
# Kaggle only
!pip install -q --upgrade torchao anthropic python-dotenv

**Only if the Gemma 4 smoke test below fails to load:** Gemma 4 needs a recent `transformers`. Run the next cell, restart the session, and run everything again from the top.

In [ ]:
# RUN ONLY IF the Gemma 4 load fails. Then restart the session and start again from the top.
# !pip install -q --upgrade transformers accelerate peft

## 2. Find the project files and prepare the working folder

**Purpose.** Copy the code, the config, the adapter and the data to `/kaggle/working/capstone7` (as Capstone 5 did with its data), load the bank, and build the same grounding for every backend: the module's curriculum text and three real examples per target.

In [2]:
import os, sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

runner = next(Path("/kaggle/input").rglob("scripts/bakeoff_runner.py"))     # the dataset can sit at any depth under /kaggle/input
sys.path.insert(0, str(runner.parent))
import bakeoff_runner as BR

project = BR.find_project()
WORK = BR.prepare_workdir(project, Path("/kaggle/working/capstone7"))
os.chdir(WORK)

from src import data as D
from src import generator as GEN
tax = GEN.load_taxonomy("data/taxonomy.csv")
raw_bank = D.load_bank("data/final_dataset.csv", tax)
configs = GEN.load_configs("config/generators.json")
curriculum = Path("data/ict_curriculum.md").read_text(encoding="utf-8")      # data/ is the working folder's own copy
grounding_for = BR.make_grounding_for(GEN, raw_bank, curriculum)

print("Project files from:", project)
print("Backends in the config:", list(configs))
found = {m: len(BR.curriculum_text(curriculum, m)) for m in "ABCDE"}
print("Curriculum text found per module (characters, at most 3000 are used; 0 means the section was not found):", found)
if not all(found.values()):
    raise RuntimeError("curriculum text missing for some module: do not start the run (the prompts would carry no curriculum reference)")

Project files from: /kaggle/input/datasets/stansir/capstone7-py
Backends in the config: ['tinyllama_lora_c5', 'gemma4_e4b_prompted', 'haiku_prompted', 'offline_cache']
Curriculum text found per module (characters, at most 3000 are used; 0 means the section was not found): {'A': 3000, 'B': 3000, 'C': 3000, 'D': 3000, 'E': 3000}


## 3. Credentials and environment

**Purpose.** Put the API key into the environment the way the key test reads it (`ANTHROPIC_API_KEY`; the base URL is the Vocareum proxy from the config), and record the library versions and GPUs for the paper.

In [3]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ["ANTHROPIC_API_KEY"] = secrets.get_secret("ANTHROPIC_API_KEY")
try:
    os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
except Exception:
    pass                                                  # optional

OUT = Path("/kaggle/working/bakeoff")
OUT.mkdir(parents=True, exist_ok=True)
env = BR.environment_report()
(OUT / "environment.json").write_text(json.dumps(env, indent=2), encoding="utf-8")
print(json.dumps(env, indent=2))

Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so


{
  "python": "3.13.15",
  "platform": "Linux-6.18.55+-x86_64-with-glibc2.39",
  "torch": "2.11.0+cu128",
  "transformers": "5.16.1",
  "peft": "0.20.0",
  "accelerate": "1.14.0",
  "anthropic": "2.3.0",
  "pandas": "2.3.3",
  "gpus": [
    {
      "name": "Tesla T4",
      "memory_gb": 15.6
    },
    {
      "name": "Tesla T4",
      "memory_gb": 15.6
    }
  ]
}


## 4. Smoke test: one candidate per backend

**Purpose.** Load each backend and write ONE candidate for the thinnest target (`d_cp_02`), so a problem (a model that does not load, a wrong key, a base URL that is blocked) shows up before the full run. Read the printed outputs: they should look like questions.

In [4]:
SMOKE = ["tinyllama_lora_c5", "haiku_prompted", "gemma4_e4b_prompted"]       # cheapest first; Gemma last
smoke = BR.smoke_test(GEN, configs, SMOKE, grounding_for)
print({name: ("OK" if r.get("ok") and r.get("valid_format") else "CHECK") for name, r in smoke.items()})

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

... (12 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

--- gemma4_e4b_prompted: ok=True valid_format=True 
    sampling used: {'temperature': 0.8, 'top_p': 0.95, 'max_new_tokens': 700, 'seed': None}
{"stem": "In a program designed to manage student records, which of the following pieces of information would most appropriately be represented by a Boolean data type?", "options": {"A": "Student's full name", "B": "Student's date of birth", "C": "Whether the student has passed the course", "D": "The student's unique identification number"}, "key": "C", "rationales": {"A": "A student's full name is represented by a string data type, as it consists of characters.", "B": "A student's date of birth is typically represented by an integer or a date/time structure, not a simple Boolean.", "C": "A Bo

{'tinyllama_lora_c5': 'OK', 'haiku_prompted': 'OK', 'gemma4_e4b_prompted': 'OK'}


### Sampling settings actually used

**Purpose.** To record how each backend was sampled, so the comparison can be reported honestly. The settings come from `config/generators.json`: TinyLlama temperature 0.8 and top-p 0.9; Gemma 4 and Haiku the defaults of the generator (Gemma: temperature 0.8, top-p 0.95; Haiku: temperature 0.7). The installed `anthropic` SDK (2.3.0) does not accept a `temperature` argument, so the generator tries `extra_body` and, if the API refuses that too, uses the API default. The cell below prints what was really applied (`applied_by`: `argument`, `extra_body` or `api_default`). The same record is saved in `summary.json`. If Haiku shows `API default`, its candidates were not sampled at 0.7 and the paper must say so.

In [5]:
print(json.dumps(GEN.SAMPLING_LOG, indent=2))

{
  "tinyllama_lora_c5": {
    "temperature": 0.8,
    "top_p": 0.9,
    "max_new_tokens": 260,
    "seed": 42
  },
  "haiku_prompted": {
    "temperature": "API default",
    "applied_by": "api_default",
    "max_tokens": 700
  },
  "gemma4_e4b_prompted": {
    "temperature": 0.8,
    "top_p": 0.95,
    "max_new_tokens": 700,
    "seed": null
  }
}


## 5. The bake-off

**Purpose.** Run the six targets, three candidates each, through every backend that loaded: 18 candidates per backend and 54 in all. One backend at a time, so a failure never loses the others; `summary.json` records the sampling used for each. The run id is part of every candidate id; set a new one (`kaggle2`, ...) in the cell for a second run, and list only the backends you want to run again.

In [6]:
# Edit these two lines when you re-run only some backends. A new run id keeps candidate ids unique.
RUN = [name for name, r in smoke.items() if r.get("ok")]          # for example ["gemma4_e4b_prompted", "haiku_prompted"]
RUN_ID = "kaggle2"                                                 # use "kaggle2", ... for a second run
print("Running:", RUN, "| run id:", RUN_ID)
summary = BR.run_bakeoff(GEN, configs, RUN, grounding_for, OUT, run_id=RUN_ID)

Running: ['tinyllama_lora_c5', 'haiku_prompted', 'gemma4_e4b_prompted'] | run id: kaggle2


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=260) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=260) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=260) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=260) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

tinyllama_lora_c5 {'candidates': 18, 'valid_format': 18, 'sampling': {'temperature': 0.8, 'top_p': 0.9, 'max_new_tokens': 260, 'seed': 42}}
haiku_prompted {'candidates': 18, 'valid_format': 18, 'sampling': {'temperature': 'API default', 'applied_by': 'api_default', 'max_tokens': 700}}


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

gemma4_e4b_prompted {'candidates': 18, 'valid_format': 18, 'sampling': {'temperature': 0.8, 'top_p': 0.95, 'max_new_tokens': 700, 'seed': None}}


## 6. Bundle the results

**Purpose.** Zip the small files that come back to the project. Download `bakeoff_bundle.zip` from the notebook's output panel. Back in the Capstone 7 notebook, Section 4.7 stores the JSONL files and runs the gate.

In [7]:
bundle = BR.bundle(OUT, "/kaggle/working/bakeoff_bundle.zip")
print(bundle, f"({bundle.stat().st_size / 1024:.0f} KB)")
for f in sorted(OUT.glob("*.jsonl")):
    lines = f.read_text(encoding="utf-8").splitlines()
    first = json.loads(lines[0])
    print(f"\n{f.name}: {len(lines)} candidates; first: {first['cand_id']} | valid format: {first['valid_format']}\n{first['raw'][:300]}")

/kaggle/working/bakeoff_bundle.zip (19 KB)

gemma4_e4b_prompted.jsonl: 18 candidates; first: GEN_gemma4_e4b_prompted_kaggle2_0001 | valid format: True
```json
{
  "stem": "In a program designed to calculate the average score of five students, the variable used to store the individual score of one student would most appropriately be defined as which data type?",
  "options": {
    "A": "Boolean",
    "B": "String",
    "C": "Integer",
    "D": "Rea

haiku_prompted.jsonl: 18 candidates; first: GEN_haiku_prompted_kaggle2_0001 | valid format: True
```json
{
  "stem": "A student is designing a program to store information about students in a school. Which of the following data should be assigned a real data type rather than an integer data type?",
  "options": {
    "A": "Student ID number",
    "B": "Year of birth",
    "C": "Grade point aver

tinyllama_lora_c5.jsonl: 18 candidates; first: GEN_tinyllama_lora_c5_kaggle2_0001 | valid format: True
What is the output of the following algorithm